In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
sys.path.insert(0,"/Users/louaness/Documents/cohesin_residence_time/ipa/utils/")
import analysis_utils as utils

# Functions for plotting

In [ ]:
def perform_analysis(pattern:list,nuclei_to_exclude:list) -> pd.DataFrame:
    """
    Perform iFRAP analysis
    :param pattern: list of strings with the pattern of the files to be analyzed
    :param nuclei_to_exclude: list of strings with the nuclei to be excluded from the analysis
    :return: DataFrame with iFRAP data + df average + df standard deviation (in time)
    """
    df = utils.concat_runs(pattern)

    utils.normalize_ifrap(df)
    df = df[~df.nucleus.isin(nuclei_to_exclude)]

    df_average = df.iloc[:, :-1].groupby('time').mean().reset_index()
    df_sd = df.iloc[:, :-1].groupby('time').std().reset_index()
    return df,df_average,df_sd

def plot_ifrap(df:pd.DataFrame,df_average:pd.DataFrame,df_sd:pd.DataFrame)->plt.Figure:
    """
    Plot iFRAP data
    :param df: DataFrame with iFRAP data
    :param df_average: DataFrame with time-average iFRAP data
    :param df_sd: DataFrame with time-standard-deviation iFRAP data
    :return: Figure with iFRAP data
    """
    a = plt.figure()
    sns.scatterplot(data=df,x='time',y='iFRAP',hue='nucleus',markers='.',linewidth=0,palette='tab20',s=8,alpha=1,legend=True)
    plt.plot(df_average.time.unique(),df_average.iFRAP,label='mean',c='black',linewidth=2, linestyle='--')
    plt.fill_between(df_average.time.unique(),df_average.iFRAP-df_sd.iFRAP,df_average.iFRAP+df_sd.iFRAP,alpha=0.5,label='std',color='grey')
    plt.xlim(4,250)
    plt.ylim(0.1,1.1)
    plt.legend(loc='upper right',bbox_to_anchor=(1.2,1.1))
    plt.xlabel('Time(frames)')
    return a 


# Import the data for NIPBL -dtag

In [ ]:
df_nipbl,df_nipbl_average,df_nipbl_sd = perform_analysis(['NIPBL', 'nodtag'],[103,123,2,151,1,112,91,0,90,241,240,242,243,244,245])
a = plot_ifrap(df_nipbl,df_nipbl_average,df_nipbl_sd)
plt.legend(loc='upper right',bbox_to_anchor=(1.22,1.0))
plt.show()

# NIPBL +dtag 

In [ ]:
df_nipbl_dtag,df_nipbl_dtag_average,df_nipbl_dtag_sd = perform_analysis(['NIPBL_FKBP', '6hdtag'],[171,172])
_ = plot_ifrap(df_nipbl_dtag,df_nipbl_dtag_average,df_nipbl_dtag_sd)

plt.xlim(0,250)
plt.show()

# WAPL -aux

In [ ]:
df_wapl,df_wapl_average,df_wapl_sd = perform_analysis(['WAPL', 'noaux'],[35,145,35])
_ = plot_ifrap(df_wapl,df_wapl_average,df_wapl_sd)
plt.xlim(0,250)
plt.ylim(0,1.1)
plt.show()

# WAPL +aux

In [ ]:
df_wapl_aux,df_wapl_aux_average,df_wapl_aux_sd = perform_analysis(['WAPL', '6h_aux','6haux'],[])
_ = plot_ifrap(df_wapl_aux,df_wapl_aux_average,df_wapl_aux_sd)
plt.xlim(0,250)

plt.show()


# WT 

In [ ]:
df_wt,df_wt_average,df_wt_sd = perform_analysis(['WT', 'RAD21'],[220])
_ = plot_ifrap(df_wt,df_wt_average,df_wt_sd)

plt.xlim(0,250)
plt.ylim(0,1.1)

# Fitting

In [ ]:
# compute offset
offset = []
for df in [df_nipbl,df_nipbl_dtag,df_wapl,df_wapl_aux,df_wt]:
    offset.append(utils.compute_offset(df))

offset = np.mean(offset)

In [ ]:
(popt_wt,pcov_wt,sd_wt), (popt_wapl,pcov_wapl,sd_wapl), (popt_nipbl,pcov_nipbl,sd_nipbl), (popt_nipbl_dtag,pcov_nipbl_dtag,sd_nipbl_dtag),(popt_wapl_aux,pcov_wapl_aux,sd_wapl_aux) = [utils.fit_df(df,offset) for df in [df_wt_average,df_wapl_average,df_nipbl_average,df_nipbl_dtag_average,df_wapl_aux_average]] 

In [ ]:
num_nuclei_condition1 = len(df_wapl_aux.nucleus.unique())
num_nuclei_condition2 = len(df_wapl.nucleus.unique())
num_nuclei_condition3 = len(df_nipbl_dtag.nucleus.unique())
num_nuclei_condition4 = len(df_nipbl.nucleus.unique())
num_nuclei_condition5 = len(df_wt.nucleus.unique())

x_wapl = df_wapl_average.time.unique()[5:]
x_wapl = x_wapl*0.5
x_wapl_aux = df_wapl_aux_average.time.unique()[5:]
x_wapl_aux = x_wapl_aux*0.5
x_nipbl = df_nipbl_average.time.unique()[5:]
x_nipbl = x_nipbl*0.5
x_nipbl_dtag = df_nipbl_dtag_average.time.unique()[5:]
x_nipbl_dtag = x_nipbl_dtag*0.5
x_wt = df_wt_average.time.unique()[5:]
x_wt = x_wt*0.5

y_wapl = df_wapl_average.iFRAP[5:]
y_wapl_aux = df_wapl_aux_average.iFRAP[5:]
y_nipbl = df_nipbl_average.iFRAP[5:]
y_nipbl_dtag = df_nipbl_dtag_average.iFRAP[5:]
y_nipbl_dtag = df_wt_average.iFRAP[5:]

plt.plot(x_wapl_aux,utils.double_exp(x_wapl_aux,*popt_wapl_aux,offset),label='fit WAPL_AID +6h aux',c='darkgreen')
plt.scatter(x_wapl_aux,y_wapl_aux,label='WAPL_AID +6h aux (average)',edgecolor='indigo',linewidth=1,facecolor='none')

plt.plot(x_wapl,utils.double_exp(x_wapl,*popt_wapl,offset),label='fit WAPL_AID - aux',c='darkred')
plt.scatter(x_wapl,y_wapl,label='WAPL_AID - aux (average)',edgecolor='black',linewidth=1,facecolor='none')

plt.plot(x_nipbl_dtag,utils.double_exp(x_nipbl_dtag,*popt_nipbl_dtag,offset),label='fit NIPBL_FKBP + 6h dtag',c='darkorange')
plt.scatter(x_nipbl_dtag,y_nipbl_dtag,label='NIPBL_FKBP + 6h dtag (average)',edgecolor='grey',linewidth=1,facecolor='none')

plt.plot(x_nipbl,utils.double_exp(x_nipbl,*popt_nipbl,offset),label='fit NIPBL_FKBP - dtag',c='navy')
plt.scatter(x_nipbl,y_nipbl,label='NIPBL_FKBP - dtag (average)',edgecolor='gold',linewidth=1,facecolor='none')

plt.plot(x_wt,utils.double_exp(x_wt,*popt_wt,offset),label='fit Rad21-Halo untreated',c='magenta')
plt.scatter(x_wt,y_nipbl_dtag,label='Rad21-Halo untreated (average)',edgecolor='saddlebrown',linewidth=1,facecolor='none')


plt.title(f' WAPL_AID +6h aux {popt_wapl_aux[0]:.2f}*exp(-{popt_wapl_aux[2]:.2f}*t) + {popt_wapl_aux[1]:.2f}*exp(-{popt_wapl_aux[3]:.2f}*t)+{0.2} (n={num_nuclei_condition1}) \n WAPL_AID - aux {popt_wapl[0]:.2f}*exp(-{popt_wapl[2]:.2f}*t) + {popt_wapl[1]:.2f}*exp(-{popt_wapl[3]:.2f}*t)+{0.2} (n={num_nuclei_condition2}) \n NIPBL_FKBP + 6h dtag {popt_nipbl_dtag[0]:.2f}*exp(-{popt_nipbl_dtag[2]:.2f}*t) + {popt_nipbl_dtag[1]:.2f}*exp(-{popt_nipbl_dtag[3]:.2f}*t)+{0.2} (n={num_nuclei_condition3}) \n NIPBL_FKBP - dtag {popt_nipbl[0]:.2f}*exp(-{popt_nipbl[2]:.2f}*t) + {popt_nipbl[1]:.2f}*exp(-{popt_nipbl[3]:.2f}*t)+{0.2} (n={num_nuclei_condition4}) \n Rad21-Halo untreated {popt_wt[0]:.2f}*exp(-{popt_wt[2]:.2f}*t) + {popt_wt[1]:.2f}*exp(-{popt_wt[3]:.2f}*t)+{0.2} (n={num_nuclei_condition5})')
plt.ylabel('iFRAP')
plt.xlabel('Time(minutes)')
plt.legend(bbox_to_anchor=(1.00, 1), loc='upper left')
# plt.savefig('../plots/iFRAP_curves.pdf',dpi=600,bbox_inches='tight')
plt.show()